# Includes

In [1]:
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.sampling import BayesianModelSampling

Bayesian Network

In [2]:
np.random.seed(42)
model = DiscreteBayesianNetwork([

('age_group','smoking_status'),
('smoking_status','smoking_years'),
('smoking_years','cumulative_smoking_exposure'),
('smoking_status','daily_cigarettes'),
('daily_cigarettes','cumulative_smoking_exposure'),

('cumulative_smoking_exposure','chronic_obstructive_pulmonary_disease'),

('environmental_pollution_level','lung_cancer_risk'),
('residential_radon_contact','lung_cancer_risk'),
('hereditary_cancer_history','lung_cancer_risk'),
('workplace_hazard_exposure','lung_cancer_risk'),
('cumulative_smoking_exposure','lung_cancer_risk'),
('chronic_obstructive_pulmonary_disease','lung_cancer_risk'),

('lung_cancer_risk','persistent_cough'),
('lung_cancer_risk','breathing_difficulty'),
('lung_cancer_risk','abnormal_chest_xray'),

('chronic_obstructive_pulmonary_disease','blood_oxygen_saturation'),
('chronic_obstructive_pulmonary_disease','lung_function_fev1')
])

In [3]:
# CPDs
cpd_age = TabularCPD('age_group', 3, [[0.3],[0.4],[0.3]])
cpd_smoker = TabularCPD(
    'smoking_status', 2,
    [[0.7,0.5,0.4],
     [0.3,0.5,0.6]],
    evidence=['age_group'],
    evidence_card=[3]
)

cpd_smoking_years = TabularCPD(
    'smoking_years', 3,
    [[0.6,0.2],
     [0.3,0.5],
     [0.1,0.3]],
    evidence=['smoking_status'],
    evidence_card=[2]
)

cpd_cigs = TabularCPD(
    'daily_cigarettes',3,
    [[0.7,0.2],
     [0.2,0.4],
     [0.1,0.4]],
    evidence=['smoking_status'],
    evidence_card=[2]
)

cpd_pack = TabularCPD(
    'cumulative_smoking_exposure',3,
    np.full((3,9),1/3),
    evidence=['smoking_years','daily_cigarettes'],
    evidence_card=[3,3]
)

cpd_copd = TabularCPD(
    'chronic_obstructive_pulmonary_disease',2,
    [[0.8,0.6,0.3],
     [0.2,0.4,0.7]],
    evidence=['cumulative_smoking_exposure'],
    evidence_card=[3]
)

cpd_pollution = TabularCPD('environmental_pollution_level',3,[[0.3],[0.4],[0.3]])
cpd_radon = TabularCPD('residential_radon_contact',2,[[0.7],[0.3]])
cpd_family = TabularCPD('hereditary_cancer_history',2,[[0.8],[0.2]])
cpd_occ = TabularCPD('workplace_hazard_exposure',2,[[0.85],[0.15]])

cpd_lung = TabularCPD(
    'lung_cancer_risk',2,
    np.full((2,144),0.5),
    evidence=[
        'cumulative_smoking_exposure',
        'chronic_obstructive_pulmonary_disease',
        'environmental_pollution_level',
        'residential_radon_contact',
        'hereditary_cancer_history',
        'workplace_hazard_exposure'
    ],
    evidence_card=[3,2,3,2,2,2]
)

cpd_cough = TabularCPD(
    'persistent_cough',2,
    [[0.8,0.3],
     [0.2,0.7]],
    evidence=['lung_cancer_risk'],
    evidence_card=[2]
)

cpd_breath = TabularCPD(
    'breathing_difficulty',2,
    [[0.8,0.4],
     [0.2,0.6]],
    evidence=['lung_cancer_risk'],
    evidence_card=[2]
)

cpd_xray = TabularCPD(
    'abnormal_chest_xray',2,
    [[0.9,0.3],
     [0.1,0.7]],
    evidence=['lung_cancer_risk'],
    evidence_card=[2]
)

cpd_oxygen = TabularCPD(
    'blood_oxygen_saturation',2,
    [[0.9,0.4],
     [0.1,0.6]],
    evidence=['chronic_obstructive_pulmonary_disease'],
    evidence_card=[2]
)

cpd_fev1 = TabularCPD(
    'lung_function_fev1',2,
    [[0.8,0.3],
     [0.2,0.7]],
    evidence=['chronic_obstructive_pulmonary_disease'],
    evidence_card=[2]
)



In [4]:
# run Model
model.add_cpds(
cpd_age,cpd_smoker,cpd_smoking_years,cpd_cigs,cpd_pack,cpd_copd,
cpd_pollution,cpd_radon,cpd_family,cpd_occ,cpd_lung,
cpd_cough,cpd_breath,cpd_xray,cpd_oxygen,cpd_fev1
)

# -------------------------
# Sample BN data
# -------------------------

sampler = BayesianModelSampling(model)
samples = sampler.forward_sample(size=5000)

# -------------------------
# Convert discrete states to realistic values
# -------------------------

def generate_age(group):
    if group == 0:
        return np.random.randint(18,40)
    elif group == 1:
        return np.random.randint(40,60)
    else:
        return np.random.randint(60,90)

samples['patient_age'] = samples['age_group'].apply(generate_age)
samples['gender'] = np.random.binomial(1,0.5,len(samples))
samples['years_of_education'] = np.random.randint(6,20,len(samples))
samples['socioeconomic_status'] = np.random.randint(1,6,len(samples))
samples['secondhand_smoke_exposure'] = np.random.binomial(1,0.3,len(samples))

samples['asthma_status'] = np.random.binomial(1,0.15,len(samples))
samples['prior_tb'] = np.random.binomial(1,0.05,len(samples))
samples['thoracic_pain'] = np.random.binomial(1,0.2,len(samples))
samples['chronic_fatigue'] = np.random.binomial(1,0.3,len(samples))

samples['body_mass_index'] = np.random.normal(26,4,len(samples)).clip(16,45)
samples['c_reactive_protein_level'] = np.random.gamma(2,2,len(samples))
samples['weekly_exercise_hours'] = np.random.normal(3,2,len(samples)).clip(0,10)

samples['nutritional_diet_quality'] = np.random.randint(1,6,len(samples))
samples['weekly_alcohol_consumption'] = np.random.poisson(3,len(samples))
samples['healthcare_service_access'] = np.random.randint(1,6,len(samples))

# Drop helper column
samples = samples.drop(columns=['age_group'])

# Reorder columns
samples = samples[[
'patient_age','gender','years_of_education','socioeconomic_status',
'smoking_status','smoking_years','daily_cigarettes',
'cumulative_smoking_exposure','secondhand_smoke_exposure',
'environmental_pollution_level','workplace_hazard_exposure',
'residential_radon_contact','hereditary_cancer_history',
'chronic_obstructive_pulmonary_disease','asthma_status',
'prior_tb','persistent_cough','thoracic_pain',
'breathing_difficulty','chronic_fatigue','body_mass_index',
'blood_oxygen_saturation','lung_function_fev1',
'c_reactive_protein_level','abnormal_chest_xray',
'weekly_exercise_hours','nutritional_diet_quality',
'weekly_alcohol_consumption','healthcare_service_access',
'lung_cancer_risk'
]]

print(samples.head())

samples.to_csv("synthetic_lung_cancer_data.csv",index=False)

  0%|          | 0/16 [00:00<?, ?it/s]

   patient_age  gender  years_of_education  socioeconomic_status  \
0           49       1                  15                     1   
1           78       1                  12                     4   
2           83       0                  19                     3   
3           52       1                  13                     5   
4           20       0                  14                     5   

   smoking_status  smoking_years  daily_cigarettes  \
0               1              1                 2   
1               0              0                 0   
2               0              0                 2   
3               1              1                 0   
4               0              0                 1   

   cumulative_smoking_exposure  secondhand_smoke_exposure  \
0                            0                          1   
1                            0                          0   
2                            0                          0   
3                     